# AgroMind — EDA and Model Training
This notebook mirrors the full ML training pipeline corresponding to `train.py`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import shap
import joblib

## 1. Load Data and Exploratory Data Analysis (EDA)

In [ ]:
DATA_PATH = '../backend/data/crop_data.csv'
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nDescribe:\n", df.describe())
print("\nNull Values:\n", df.isnull().sum())

In [ ]:
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

## 2. Preprocessing
Standard scaling for numerical stability and label encoding for the target string categorical values.

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 3. Training Classification Models
Training all classifiers and building a soft voting ensemble.

In [ ]:
models = {
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, learning_rate=0.1, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
}

voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=200, learning_rate=0.1, use_label_encoder=False, eval_metric='mlogloss', random_state=42)),
        ('svm', SVC(kernel='rbf', probability=True, random_state=42))
    ],
    voting='soft'
)
models['Voting Ensemble'] = voting_clf

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    results.append({'Model': name, 'Accuracy': acc, 'F1': f1})
    print(f"{name} - Accuracy: {acc:.4f}, F1: {f1:.4f}")


## 4. Synthesizing Yield and Linear Regression
Since the original dataset lacks yield data, we generate a synthetic target.

In [ ]:
df['synthetic_yield'] = (df['N'] * 0.4) + (df['K'] * 0.3) + (df['rainfall'] * 0.002) + np.random.normal(0, 10, size=len(df))
df['synthetic_yield'] = df['synthetic_yield'].apply(lambda x: max(x, 10))
X_yield = df.drop(['label', 'synthetic_yield'], axis=1)
y_yield = df['synthetic_yield']
X_yield_train, X_yield_test, y_yield_train, y_yield_test = train_test_split(X_yield, y_yield, test_size=0.2, random_state=42)
yield_scaler = StandardScaler()
X_yts = yield_scaler.fit_transform(X_yield_train)
yield_model = LinearRegression()
yield_model.fit(X_yts, y_yield_train)
print("Linear Regression trained for Yield Estimation.")

## 5. K-Means Clustering
For soil profile categorization, based on 5 theoretical profiles.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(scaler.transform(X))
print("KMeans Clustering trained.")

## 6. SHAP Value Feature Importance

In [ ]:
explainer = shap.TreeExplainer(models['Random Forest'])
shap_values = explainer.shap_values(X_train_scaled[:100])
shap.summary_plot(shap_values, X_train_scaled[:100], feature_names=X.columns)

---
## Part 2: CNN Soil Classification


This section trains a custom 4-block Convolutional Neural Network **from scratch** (no pretrained weights, no transfer learning) to classify soil images into 8 soil types.

The CNN output feeds an agronomic lookup table (NPK/pH midpoints), which combines with region climate data to form the full 7-feature vector passed to the ensemble recommender above.

### Dataset
- **Source**: Soil Image Dataset (Kaggle) by jayaprakashpondy
- **URL**: https://www.kaggle.com/datasets/jayaprakashpondy/soil-image-dataset
- **~4,600 images** | **8 classes**: alluvial, black, clay, red, sandy, loamy, laterite, chalky

```bash
# Download via Kaggle CLI
kaggle datasets download -d jayaprakashpondy/soil-image-dataset
unzip soil-image-dataset.zip -d ../backend/data/soil_images/
```

In [ ]:
# CNN Imports
import os, json, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, Activation, MaxPooling2D,
    GlobalAveragePooling2D, Dense, Dropout
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

DATA_DIR = '../backend/data/soil_images/'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32


### 2.1 Explore Dataset — Sample Images per Soil Class

In [ ]:
SOIL_CLASSES = ['Alluvial Soil','Black Soil','Clay Soil','Red Soil',
                'Sandy Soil','Loamy Soil','Laterite Soil','Chalky Soil']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Images per Soil Class', fontsize=16, fontweight='bold')

for ax, cls in zip(axes.flatten(), SOIL_CLASSES):
    cls_dir = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_dir):
        imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if imgs:
            img = Image.open(os.path.join(cls_dir, random.choice(imgs))).resize((224,224))
            ax.imshow(img)
            ax.set_title(cls.replace(' Soil',''), fontsize=10, fontweight='bold')
        else:
            ax.text(0.5,0.5,'No images',ha='center',va='center',transform=ax.transAxes)
    else:
        ax.text(0.5,0.5,'Path not found',ha='center',va='center',transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout(); plt.show()

print('\nClass Image Counts:')
for cls in SOIL_CLASSES:
    d = os.path.join(DATA_DIR, cls)
    count = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'  {cls:20s}: {count} images')


### 2.2 CNN Architecture (from scratch — no pretrained weights)


In [ ]:
def build_cnn(num_classes=8):
    """
    Custom 4-block CNN — built from scratch.
    No pretrained weights (no ImageNet, no transfer learning).
    """
    model = Sequential([
        # Block 1
        Conv2D(32, (3,3), padding='same', input_shape=(224,224,3)),
        BatchNormalization(), Activation('relu'), MaxPooling2D(2,2),
        # Block 2
        Conv2D(64, (3,3), padding='same'),
        BatchNormalization(), Activation('relu'), MaxPooling2D(2,2),
        # Block 3
        Conv2D(128, (3,3), padding='same'),
        BatchNormalization(), Activation('relu'), MaxPooling2D(2,2),
        # Block 4
        Conv2D(256, (3,3), padding='same'),
        BatchNormalization(), Activation('relu'), GlobalAveragePooling2D(),
        # Head
        Dense(256, activation='relu'), Dropout(0.4),
        Dense(num_classes, activation='softmax'),
    ], name='soil_cnn_v2')
    return model

model = build_cnn()
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


### 2.3 Data Generators (80 / 10 / 10 split)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=0.2,
    height_shift_range=0.2, horizontal_flip=True, zoom_range=0.2,
    brightness_range=[0.8,1.2], validation_split=0.2
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.5)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True, seed=42
)
val_gen = val_datagen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False, seed=42
)
print(f'Train: {train_gen.samples} | Val: {val_gen.samples}')
print(f'Classes: {list(train_gen.class_indices.keys())}')


### 2.4 Train CNN

In [ ]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-6),
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_gen, validation_data=val_gen, epochs=50, callbacks=callbacks, verbose=1
)

val_gen.reset()
loss, acc = model.evaluate(val_gen, verbose=0)
print(f'\n=== Final Test Accuracy: {acc*100:.2f}% | Loss: {loss:.4f} ===')


### 2.5 Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN Soil Classifier — Training History', fontsize=14, fontweight='bold')

ax1.plot(history.history['accuracy'], label='Train', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val', linewidth=2, linestyle='--')
ax1.set_title('Accuracy Curves'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_ylim([0,1])

ax2.plot(history.history['loss'], label='Train', linewidth=2, color='coral')
ax2.plot(history.history['val_loss'], label='Val', linewidth=2, linestyle='--', color='darkred')
ax2.set_title('Loss Curves'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


### 2.6 Confusion Matrix Heatmap

In [ ]:
val_gen.reset()
y_pred = np.argmax(model.predict(val_gen, verbose=0), axis=1)
y_true = val_gen.classes
idx_to_class = {v: k for k, v in train_gen.class_indices.items()}
class_names = [idx_to_class[i] for i in sorted(idx_to_class)]

print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-8)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, linewidths=0.5)
plt.title('CNN Confusion Matrix — 8 Soil Classes (Normalised)', fontsize=13, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

# Save model + class indices
model.save('../backend/models/saved/soil_cnn.h5')
with open('../backend/models/saved/class_indices.json', 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print('Model saved → backend/models/saved/soil_cnn.h5')
